In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/sk48/d8078629/sk48.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/sk48/d8078629/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/tn36/ef4dde99/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/tn36/ef4dde99/tn36.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/m0r0/492f87ba/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/m0r0/492f87ba/m0r0.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/bp35/0a0ad940/bp35.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/bp35/0a0ad940/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/cn04/2fe56bfb/cn04.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/cn04/2fe56bfb/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-

In [2]:
import os
import subprocess

subprocess.run(["pip", "install", "ftfy", "regex", "tqdm", "opencv-python-headless", 
                "facenet-pytorch"], capture_output=True)

subprocess.run(["pip", "install", "git+https://github.com/openai/CLIP.git"], capture_output=True)

CompletedProcess(args=['pip', 'install', 'git+https://github.com/openai/CLIP.git'], returncode=0, stdout=b"Collecting git+https://github.com/openai/CLIP.git\n  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-spykhnr1\n  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6\n  Preparing metadata (setup.py): started\n  Preparing metadata (setup.py): finished with status 'done'\nRequirement already satisfied: ftfy in /usr/local/lib/python3.12/dist-packages (from clip==1.0) (6.3.1)\nRequirement already satisfied: packaging in /usr/local/lib/python3.12/dist-packages (from clip==1.0) (26.1)\nRequirement already satisfied: regex in /usr/local/lib/python3.12/dist-packages (from clip==1.0) (2025.11.3)\nRequirement already satisfied: tqdm in /usr/local/lib/python3.12/dist-packages (from clip==1.0) (4.67.3)\nRequirement already satisfied: torch in /usr/local/lib/python3.12/dist-packages (from clip==1.0) (2.10.0+cu128)\nRequirement already 

In [3]:
import torch
import clip
import cv2
import csv
import json
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
from facenet_pytorch import MTCNN
 
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [4]:
KAGGLE_INPUT = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"

ZIP1 = os.path.join(KAGGLE_INPUT, "Hi-EF-20260829T071606Z-1-001", "Hi-EF")
ZIP2 = os.path.join(KAGGLE_INPUT, "Hi-EF-20260829T071606Z-1-002", "Hi-EF")
 
# CSV paths (all in zip 1)
sample_csv_path = os.path.join(ZIP1, "sample.csv")
annotation_csv_path = os.path.join(ZIP1, "annotation.csv")
 
# Video directories (split across both zips)
video_roots = [
    os.path.join(ZIP1, "video"),
    os.path.join(ZIP2, "video"),
]
 
# Audio directories (split across both zips)
audio_roots = [
    os.path.join(ZIP1, "audio"),
    os.path.join(ZIP2, "audio"),
]

In [5]:
def find_video(clip_id):
    """Find a video file across both zip directories.
    clip_id is like '01/00142' → look for video/01/00142.mp4
    """
    for root in video_roots:
        path = os.path.join(root, clip_id + ".mp4")
        if os.path.exists(path):
            return path
    return None
 
# Verify paths
print(f"sample.csv exists:     {os.path.exists(sample_csv_path)}")
print(f"annotation.csv exists: {os.path.exists(annotation_csv_path)}")
 
for i, vr in enumerate(video_roots):
    if os.path.exists(vr):
        shows = sorted(os.listdir(vr))
        print(f"Video root {i+1}: {len(shows)} show folders — {shows[:5]}...")
    else:
        print(f"Video root {i+1}: NOT FOUND at {vr}")
 
# Count total videos across both zips
total_videos = 0
for vr in video_roots:
    if os.path.exists(vr):
        for show in os.listdir(vr):
            show_path = os.path.join(vr, show)
            if os.path.isdir(show_path):
                total_videos += len([f for f in os.listdir(show_path) if f.endswith('.mp4')])
print(f"\nTotal .mp4 files across both zips: {total_videos}")

sample.csv exists:     True
annotation.csv exists: True
Video root 1: 52 show folders — ['01', '02', '03', '04', '05']...
Video root 2: 23 show folders — ['01', '04', '19', '23', '30']...

Total .mp4 files across both zips: 7925


In [6]:
samples = []
with open(sample_csv_path, 'r') as f:
    reader = csv.reader(f)
    header = next(reader)
    for row in reader:
        samples.append(row)
 
print(f"Total MCIS samples: {len(samples)}")
print(f"Header: {header}")
print(f"Example row: {samples[0]}")
 
# Collect ALL unique clip IDs across all 4 positions
all_clip_ids = set()
for s in samples:
    for i in range(1, 5):  # columns 1-4 are clip IDs
        all_clip_ids.add(s[i].strip())
 
print(f"\nTotal unique clip IDs to process: {len(all_clip_ids)}")
 
# %%
# Load annotations (for text/utterances)
annotations = {}
with open(annotation_csv_path, 'r') as f:
    for row in csv.reader(f):
        if len(row) >= 2:
            clip_id = row[0].strip()
            utterance = row[1].strip() if len(row) > 1 else ""
            annotations[clip_id] = {
                'text': utterance,
                'emotion': row[7].strip() if len(row) > 7 and row[7].strip() else None,
                'polarity': row[5].strip() if len(row) > 5 and row[5].strip() else None,
                'intensity': row[6].strip() if len(row) > 6 and row[6].strip() else None,
                'uncertainty': row[8].strip() if len(row) > 8 and row[8].strip() else None,
            }
 
clips_with_text = sum(1 for cid in all_clip_ids if cid in annotations and annotations[cid]['text'])
clips_with_emotion = sum(1 for cid in all_clip_ids if cid in annotations and annotations[cid]['emotion'])
print(f"Clips with utterance text: {clips_with_text}/{len(all_clip_ids)}")
print(f"Clips with emotion label: {clips_with_emotion}/{len(all_clip_ids)}")

Total MCIS samples: 2830
Header: ['sample_name', 'clip1', 'clip2', 'clip3', 'clip4']
Example row: ['sample00001', '01/00059', '01/00060', '01/00061', '01/00062']

Total unique clip IDs to process: 7925
Clips with utterance text: 7905/7925
Clips with emotion label: 4783/7925


In [7]:
clip_model, clip_preprocess = clip.load("ViT-B/32", device=DEVICE)
clip_model.eval()
print("CLIP ViT-B/32 loaded")
 
# Load MTCNN for face detection
mtcnn = MTCNN(
    image_size=224,       # output face crop size (CLIP input)
    margin=20,            # margin around detected face
    min_face_size=40,
    thresholds=[0.6, 0.7, 0.7],
    post_process=False,   # don't normalize, we'll use CLIP's preprocessing
    device=DEVICE
)
print("MTCNN face detector loaded")
 
print(f"\nCLIP input resolution: {clip_preprocess.transforms[0].size}")

100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 116MiB/s]


CLIP ViT-B/32 loaded
MTCNN face detector loaded

CLIP input resolution: 224


In [8]:
def sample_frames(video_path, n_frames=16):
    """Sample n_frames uniformly from a video file.
    
    Returns:
        frames: list of PIL Images (RGB)
        success: bool
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return [], False
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return [], False
    
    # Uniform sampling: pick n_frames evenly spaced indices
    if total_frames >= n_frames:
        indices = np.linspace(0, total_frames - 1, n_frames, dtype=int)
    else:
        # Fewer frames than needed — take all and pad by repeating last
        indices = list(range(total_frames)) + [total_frames - 1] * (n_frames - total_frames)
    
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            # BGR → RGB → PIL
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame_rgb))
        else:
            # If read fails, duplicate last successful frame
            if frames:
                frames.append(frames[-1].copy())
            else:
                frames.append(Image.new('RGB', (224, 224), (0, 0, 0)))
    
    cap.release()
    return frames[:n_frames], True
 
 
def detect_and_crop_faces(frames, mtcnn_model):
    """Detect faces in frames and return cropped face images.
    
    For frames where no face is detected, returns a center crop as fallback.
    
    Returns:
        face_crops: list of PIL Images (224x224)
        valid_mask: list of bools (True = face detected)
    """
    face_crops = []
    valid_mask = []
    
    for frame in frames:
        # Try MTCNN detection
        try:
            boxes, probs = mtcnn_model.detect(frame)
            if boxes is not None and len(boxes) > 0:
                # Take the highest-confidence face
                best_idx = probs.argmax()
                box = boxes[best_idx].astype(int)
                
                # Crop with bounds checking
                w, h = frame.size
                x1 = max(0, box[0])
                y1 = max(0, box[1])
                x2 = min(w, box[2])
                y2 = min(h, box[3])
                
                if (x2 - x1) > 10 and (y2 - y1) > 10:
                    face = frame.crop((x1, y1, x2, y2))
                    face = face.resize((224, 224), Image.BILINEAR)
                    face_crops.append(face)
                    valid_mask.append(True)
                    continue
        except Exception:
            pass
        
        # Fallback: center crop
        w, h = frame.size
        crop_size = min(w, h)
        left = (w - crop_size) // 2
        top = (h - crop_size) // 2
        center_crop = frame.crop((left, top, left + crop_size, top + crop_size))
        center_crop = center_crop.resize((224, 224), Image.BILINEAR)
        face_crops.append(center_crop)
        valid_mask.append(False)
    
    return face_crops, valid_mask
 
 
@torch.no_grad()
def encode_frames_clip(frames, clip_model, clip_preprocess, device):
    """Encode a list of PIL images through CLIP's vision encoder.
    
    Returns:
        features: tensor of shape [n_frames, 512]
    """
    images = torch.stack([clip_preprocess(f) for f in frames]).to(device)
    features = clip_model.encode_image(images)
    features = features.float()  # ensure float32
    # L2 normalize (consistent with CLIP's default)
    features = features / features.norm(dim=-1, keepdim=True)
    return features.cpu()
 
 
@torch.no_grad()
def encode_text_clip(text, clip_model, device):
    """Encode a text string through CLIP's text encoder.
    
    Returns:
        feature: tensor of shape [512]
    """
    if not text or text.strip() == "":
        return torch.zeros(512)
    
    # CLIP tokenizer truncates to 77 tokens
    tokens = clip.tokenize([text], truncate=True).to(device)
    feature = clip_model.encode_text(tokens)
    feature = feature.float()
    feature = feature / feature.norm(dim=-1, keepdim=True)
    return feature.squeeze(0).cpu()

In [9]:
OUTPUT_DIR = "/kaggle/working/hi_ef_features"
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
N_FRAMES = 16
BATCH_LOG_INTERVAL = 100
 
# Sort for deterministic ordering
clip_ids_sorted = sorted(all_clip_ids)
 
# Track progress and failures
results = {
    'success': 0,
    'video_not_found': 0,
    'video_read_failed': 0,
    'errors': []
}
 
print(f"Extracting features for {len(clip_ids_sorted)} clips...")
print(f"Output: {OUTPUT_DIR}")
print(f"Frames per clip: {N_FRAMES}")
print()
 
for idx, clip_id in enumerate(tqdm(clip_ids_sorted, desc="Extracting")):
    # Check if already processed
    out_path = os.path.join(OUTPUT_DIR, clip_id.replace('/', '_') + '.pt')
    if os.path.exists(out_path):
        results['success'] += 1
        continue
    
    # Locate video file (search both zip directories)
    video_path = find_video(clip_id)
    if video_path is None:
        results['video_not_found'] += 1
        if len(results['errors']) < 20:
            results['errors'].append(f"Not found: {clip_id}.mp4 in either zip")
        continue
    
    try:
        # 1. Sample frames
        frames, success = sample_frames(video_path, N_FRAMES)
        if not success or len(frames) == 0:
            results['video_read_failed'] += 1
            continue
        
        # 2. Detect faces and crop
        face_crops, face_valid_mask = detect_and_crop_faces(frames, mtcnn)
        
        # 3. Encode full frames through CLIP
        ori_features = encode_frames_clip(frames, clip_model, clip_preprocess, DEVICE)
        
        # 4. Encode face crops through CLIP
        face_features = encode_frames_clip(face_crops, clip_model, clip_preprocess, DEVICE)
        
        # 5. Encode text (utterance)
        text = ""
        if clip_id in annotations:
            text = annotations[clip_id].get('text', '')
        text_feature = encode_text_clip(text, clip_model, DEVICE)
        
        # 6. Get annotation metadata (if available)
        meta = {}
        if clip_id in annotations:
            ann = annotations[clip_id]
            meta = {
                'emotion': ann.get('emotion'),
                'polarity': ann.get('polarity'),
                'intensity': ann.get('intensity'),
                'uncertainty': ann.get('uncertainty'),
                'text': text,
            }
        
        # 7. Save everything as one .pt file
        output = {
            'clip_id': clip_id,
            'ori_features': ori_features,           # [16, 512] full frame CLIP features
            'face_features': face_features,          # [16, 512] face crop CLIP features
            'face_valid_mask': face_valid_mask,       # [16] bool, True = face was detected
            'text_feature': text_feature,             # [512] CLIP text feature
            'meta': meta,                             # annotation metadata
        }
        
        torch.save(output, out_path)
        results['success'] += 1
        
    except Exception as e:
        if len(results['errors']) < 20:
            results['errors'].append(f"Error on {clip_id}: {str(e)}")
        continue
    
    # Log progress
    if (idx + 1) % BATCH_LOG_INTERVAL == 0:
        print(f"\n[{idx+1}/{len(clip_ids_sorted)}] "
              f"Success: {results['success']}, "
              f"Not found: {results['video_not_found']}, "
              f"Read failed: {results['video_read_failed']}"

SyntaxError: incomplete input (3488278826.py, line 95)

In [ ]:
print("=== Extraction Summary ===")
print(f"Total clips to process:  {len(clip_ids_sorted)}")
print(f"Successfully extracted:  {results['success']}")
print(f"Video not found:         {results['video_not_found']}")
print(f"Video read failed:       {results['video_read_failed']}")
print()
 
if results['errors']:
    print("=== First 10 Errors ===")
    for e in results['errors'][:10]:
        print(f"  {e}")
    print()
 
# Verify a random sample
extracted_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.pt')]
print(f"Total .pt files saved: {len(extracted_files)}")
 
if extracted_files:
    # Load and inspect one
    sample_file = os.path.join(OUTPUT_DIR, extracted_files[0])
    sample_data = torch.load(sample_file, map_location='cpu', weights_only=False)
    
    print(f"\n=== Sample: {extracted_files[0]} ===")
    for key, val in sample_data.items():
        if isinstance(val, torch.Tensor):
            print(f"  {key}: shape={val.shape}, dtype={val.dtype}")
        elif isinstance(val, list):
            print(f"  {key}: list of {len(val)}, e.g. {val[:3]}")
        elif isinstance(val, dict):
            print(f"  {key}: {val}")
        else:
            print(f"  {key}: {val}")
 
# %%
# Check MCIS coverage — how many MCIS have ALL 4 clips extracted?
extracted_ids = set(f.replace('_', '/', 1).replace('.pt', '') for f in extracted_files)
 
full_mcis = 0
partial_mcis = 0
missing_mcis = 0
 
for s in samples:
    clips = [s[i].strip() for i in range(1, 5)]
    found = sum(1 for c in clips if c in extracted_ids)
    if found == 4:
        full_mcis += 1
    elif found > 0:
        partial_mcis += 1
    else:
        missing_mcis += 1
 
print(f"\n=== MCIS Coverage ===")
print(f"Full (all 4 clips):  {full_mcis}/{len(samples)} ({100*full_mcis/len(samples):.1f}%)")
print(f"Partial (1-3 clips): {partial_mcis}/{len(samples)}")
print(f"Missing (0 clips):   {missing_mcis}/{len(samples)}")

In [ ]:
import shutil
 
output_zip = "/kaggle/working/hi_ef_features"
shutil.make_archive(output_zip, 'zip', OUTPUT_DIR)
zip_size = os.path.getsize(output_zip + '.zip') / 1e6
print(f"Saved: {output_zip}.zip ({zip_size:.1f} MB)")
print("Download this file or create a Kaggle dataset from it for use in training notebooks.")

In [ ]:
mcis_index = []
 
emotion_map = {
    'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 
    'neutral': 4, 'sad': 5, 'surprise': 6
}
polarity_map = {'positive': 0, 'neutral': 1, 'negative': 2}
intensity_map = {'weak': 0, 'powerful': 1}
 
for s in samples:
    sample_id = s[0].strip()
    clips = [s[i].strip() for i in range(1, 5)]
    
    entry = {
        'sample_id': sample_id,
        'clip_ids': clips,
        'feature_files': [c.replace('/', '_') + '.pt' for c in clips],
    }
    
    # Add labels for clip III (Party A) and clip IV (Party B)
    for label_name, clip_idx in [('clip3', 2), ('clip4', 3)]:
        cid = clips[clip_idx]
        if cid in annotations and annotations[cid]['emotion']:
            ann = annotations[cid]
            entry[f'{label_name}_emotion'] = emotion_map.get(ann['emotion'], -1)
            entry[f'{label_name}_emotion_str'] = ann['emotion']
            entry[f'{label_name}_polarity'] = polarity_map.get(ann.get('polarity', ''), -1)
            entry[f'{label_name}_intensity'] = intensity_map.get(ann.get('intensity', ''), -1)
            entry[f'{label_name}_uncertainty'] = int(ann['uncertainty']) if ann.get('uncertainty', '').isdigit() else -1
        else:
            entry[f'{label_name}_emotion'] = -1
            entry[f'{label_name}_emotion_str'] = None
            entry[f'{label_name}_polarity'] = -1
            entry[f'{label_name}_intensity'] = -1
            entry[f'{label_name}_uncertainty'] = -1
    
    mcis_index.append(entry)
 
# Save
index_path = os.path.join(OUTPUT_DIR, 'mcis_index.json')
with open(index_path, 'w') as f:
    json.dump(mcis_index, f, indent=2)
 
print(f"Saved MCIS index: {index_path}")
print(f"Total entries: {len(mcis_index)}")
 
# Preview
print("\n=== Sample entry ===")
print(json.dumps(mcis_index[0], indent=2))